# Phishing Detection

## What This Is
Phishing detection classifies URLs, emails, and web pages as legitimate or phishing. Modern approaches combine multiple signal types:

- **URL-based features**: Length, number of subdomains, presence of IP in URL, special characters, TLD reputation, brand name impersonation
- **HTML features**: Hidden elements, form action pointing to different domain, iframe count, JavaScript obfuscation
- **Visual similarity**: Render the page, compare screenshot to legitimate brand pages (perceptual hashing, CLIP embeddings)
- **Lexical analysis**: Edit distance to known brands in the domain name
- **Certificate transparency**: Newly registered domains, short certificate validity

## Real-World Deployment
Production phishing detection runs in layers:
1. URL reputation lists (fast, but reactive)
2. ML classifier on URL features (fast, catches new phishing)
3. HTML + visual analysis for high-confidence verdicts (slow, high accuracy)
4. Manual review for escalations

In [ ]:
import numpy as np
import re
from typing import List, Dict, Tuple

np.random.seed(42)

# Feature extraction from URL strings

SUSPICIOUS_KEYWORDS = [
    'login', 'signin', 'account', 'verify', 'secure', 'update',
    'confirm', 'banking', 'paypal', 'apple', 'microsoft', 'amazon',
    'password', 'credential'
]

LEGITIMATE_TLDS = {'.com', '.org', '.net', '.edu', '.gov'}
SUSPICIOUS_TLDS = {'.xyz', '.top', '.click', '.loan', '.gq', '.tk', '.ml', '.ga'}

def extract_url_features(url: str) -> np.ndarray:
    """Extract URL-based phishing features."""
    url_lower = url.lower()
    
    # Remove protocol
    domain_part = url_lower.replace('https://', '').replace('http://', '')
    path_start = domain_part.find('/')
    domain = domain_part[:path_start] if path_start > 0 else domain_part
    path = domain_part[path_start:] if path_start > 0 else '/'
    
    features = [
        len(url),                                          # URL length
        len(domain),                                       # domain length
        domain.count('.'),                                 # subdomains
        url_lower.count('-'),                              # hyphen count
        url_lower.count('@'),                              # @ symbol
        url_lower.count('//'),                             # double slash
        int(bool(re.search(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', domain))),  # IP in domain
        sum(1 for kw in SUSPICIOUS_KEYWORDS if kw in url_lower),  # suspicious keywords
        int(any(url_lower.endswith(tld) or ('.' + tld.strip('.') + '/') in url_lower 
                for tld in SUSPICIOUS_TLDS)),              # suspicious TLD
        int('https' in url_lower),                        # has HTTPS
        len(path),                                         # path length
        path.count('/'),                                   # path depth
        url_lower.count('%'),                              # URL encoding
        int(domain.count('-') > 2),                        # many hyphens
        int(len(domain) > 30),                             # long domain
    ]
    return np.array(features, dtype=float)

# Generate sample URLs
phishing_urls = [
    'http://paypal-secure-login.xyz/verify/account?id=12345',
    'http://192.168.1.1/login/microsoft/credential/update',
    'http://apple-id-verify-secure.tk/signin/confirm',
    'http://amazon-account-suspended.click/billing/update',
    'http://login-banking-secure-update.ml/account/verify',
]

legit_urls = [
    'https://www.paypal.com/signin',
    'https://microsoft.com/en-us/account',
    'https://appleid.apple.com/signin',
    'https://amazon.com/account/homepage',
    'https://bankofamerica.com/online/login/',
]

print('URL Feature Extraction:')
print(f'{'URL':<50} {'Features'}')
for url in phishing_urls[:2]:
    feats = extract_url_features(url)
    print(f'  [PHISH] {url[:45]:50s} len={feats[0]:.0f} subdoms={feats[2]:.0f} keywords={feats[7]:.0f}')
for url in legit_urls[:2]:
    feats = extract_url_features(url)
    print(f'  [LEGIT] {url[:45]:50s} len={feats[0]:.0f} subdoms={feats[2]:.0f} keywords={feats[7]:.0f}')


In [ ]:
# Generate full URL dataset and train classifier

def generate_phishing_url() -> str:
    brands = ['paypal', 'apple', 'amazon', 'microsoft', 'google', 'banking']
    actions = ['login', 'verify', 'secure', 'update', 'confirm', 'account']
    tlds = ['.xyz', '.top', '.click', '.ml', '.tk', '.ga']
    brand = np.random.choice(brands)
    action = np.random.choice(actions)
    tld = np.random.choice(tlds)
    n_extra = np.random.randint(0, 3)
    extra = '-'.join(np.random.choice(actions, n_extra))
    domain = f'{brand}-{action}'
    if extra: domain += f'-{extra}'
    domain += tld
    path_parts = np.random.choice(actions, np.random.randint(1, 4))
    path = '/' + '/'.join(path_parts)
    return f'http://{domain}{path}'

def generate_legit_url() -> str:
    brands = ['paypal.com', 'apple.com', 'amazon.com', 'microsoft.com', 'google.com']
    paths = ['/signin', '/account', '/products', '/support', '/download']
    brand = np.random.choice(brands)
    path = np.random.choice(paths)
    sub = np.random.choice(['www', 'id', 'accounts', 'shop', ''])
    domain = f'{sub}.{brand}' if sub else brand
    return f'https://{domain}{path}'

N = 1000
phish_X = np.array([extract_url_features(generate_phishing_url()) for _ in range(N)])
legit_X = np.array([extract_url_features(generate_legit_url()) for _ in range(N)])

X = np.vstack([phish_X, legit_X])
y = np.array([1]*N + [0]*N)

# Shuffle and split
perm = np.random.permutation(len(X))
X, y = X[perm], y[perm]
split = 1600
X_tr, y_tr = X[:split], y[:split]
X_te, y_te = X[split:], y[split:]

def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

mu, s = X_tr.mean(0), X_tr.std(0) + 1
X_tr_n = (X_tr - mu) / s
X_te_n = (X_te - mu) / s

w = np.zeros(X_tr_n.shape[1]); b = 0.0
for _ in range(300):
    e = sigmoid(X_tr_n @ w + b) - y_tr
    w -= 0.05 * (X_tr_n.T @ e) / len(y_tr)
    b -= 0.05 * e.mean()

pred = (sigmoid(X_te_n @ w + b) > 0.5).astype(int)
tp = ((pred==1)&(y_te==1)).sum(); fp = ((pred==1)&(y_te==0)).sum()
fn = ((pred==0)&(y_te==1)).sum(); tn = ((pred==0)&(y_te==0)).sum()
prec = tp/(tp+fp+1e-8); rec = tp/(tp+fn+1e-8)
print('Phishing URL Classifier:')
print(f'  Accuracy:  {(pred==y_te).mean():.3f}')
print(f'  Precision: {prec:.3f}')
print(f'  Recall:    {rec:.3f}')
print(f'  F1:        {2*prec*rec/(prec+rec+1e-8):.3f}')

FEAT_NAMES = ['url_len','domain_len','subdoms','hyphens','at_sign','dbl_slash','ip_in_dom',
              'susp_kw','susp_tld','has_https','path_len','path_depth','url_enc','many_hyphens','long_dom']
top5 = np.argsort(np.abs(w))[::-1][:5]
print('Top 5 features:', [FEAT_NAMES[i] for i in top5])
